In [ ]:
import os
import sys
from datetime import datetime
today = datetime.today().strftime('%Y-%m-%d')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import hdf5plugin
import decoupler as dc

import warnings
warnings.filterwarnings('ignore', category=ad.ImplicitModificationWarning)

# Add repo path to sys path (allows to access scripts and metadata from repo)
repo_path,_ = os.path.split(os.path.split(os.getcwd())[0])
repo_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis'
sys.path.insert(1, repo_path) 
sys.path.insert(2, '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts')

# Autoreload custom scripts
%load_ext autoreload
%autoreload 2

# Define paths
plots_path = f'{repo_path}/plots/'
data_path = f'{repo_path}/data/'
model_path = os.path.join(repo_path, 'models')
general_data_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/data'

print('Dir for plots: {}'.format(plots_path))
print('Dir for data: {}'.format(data_path))

# Formatting
from matplotlib import font_manager
font_manager.fontManager.addfont("/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf")
plt.style.use('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts/plotting/thyAgeing.mplstyle')

In [ ]:
collectri = dc.op.collectri(organism="human")
collectri

In [ ]:
collectri_ar = collectri.loc[collectri['source'] == 'AR']
collectri_ar

In [ ]:
progeny = dc.op.progeny(organism="human")
progeny

In [ ]:
progeny_ar = progeny.loc[progeny['source'] == 'Androgen']

In [ ]:
len(progeny_ar), len(collectri_ar)

In [ ]:
# Compare target membership between the two dataframes
progeny_targets = set(progeny_ar["target"].astype(str))
collectri_targets = set(collectri_ar["target"].astype(str))

all_targets = sorted(progeny_targets | collectri_targets)

cm_df = pd.DataFrame({
    "progeny_ar": [t in progeny_targets for t in all_targets],
    "collectri_ar": [t in collectri_targets for t in all_targets],
})

conf_mat = pd.crosstab(cm_df["progeny_ar"], cm_df["collectri_ar"])
conf_mat.index = ["Not in progeny_ar", "In progeny_ar"]
conf_mat.columns = ["Not in collectri_ar", "In collectri_ar"]

plt.figure(figsize=(6, 5))
sns.heatmap(conf_mat, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion matrix of target presence\nprogeny_ar vs collectri_ar")
plt.xlabel("collectri_ar")
plt.ylabel("progeny_ar")
plt.tight_layout()
plt.show()

In [ ]:
degs = pd.read_csv("/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/data/analyses/dea/thyAgeing_dea_taa_l4_adult_vs_infant_ageEffect.csv")
degs.head()

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

degs_ar = degs[degs['gene_name'].isin(all_targets)]

import matplotlib.pyplot as plt

# Pivot the data for heatmap
heatmap_data = degs_ar.pivot_table(index='gene_name', columns='cell_type', values='log2FoldChange', fill_value=0)

# Calculate linkage for clustering
row_linkage = linkage(pdist(heatmap_data, metric='euclidean'), method='ward')
col_linkage = linkage(pdist(heatmap_data.T, metric='euclidean'), method='ward')

# Create clustered heatmap
plt.figure(figsize=(14, 30))
g = sns.clustermap(
    heatmap_data,
    cmap='RdBu_r', center=0, vmin=-2, vmax=2,
    dendrogram_ratio=(0.2, 0.05),
    row_linkage=row_linkage,
    col_linkage=col_linkage,
    figsize=(14, 30),
    cbar_kws={'label': 'log2FoldChange'}
)
g.savefig(f'{plots_path}/hormoneSignalling/hormone_signalling_joint_ar_targets_deg_l4_heatmap.pdf', dpi=300)

In [ ]:
degs_ar_baseMean = degs_ar.groupby('gene_name').agg(
    baseMean = ('baseMean', 'max'))
degs_ar_baseMean.baseMean.describe()

In [ ]:
exclude_genes = degs_ar_baseMean[degs_ar_baseMean.baseMean < 20].index.tolist()

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

degs_ar = degs[degs['gene_name'].isin(all_targets)]
degs_ar = degs_ar[~degs_ar['gene_name'].isin(exclude_genes)]

import matplotlib.pyplot as plt

# Pivot the data for heatmap
heatmap_data = degs_ar.pivot_table(index='gene_name', columns='cell_type', values='log2FoldChange', fill_value=0)

# Calculate linkage for clustering
row_linkage = linkage(pdist(heatmap_data, metric='euclidean'), method='ward')
col_linkage = linkage(pdist(heatmap_data.T, metric='euclidean'), method='ward')

# Create clustered heatmap
plt.figure(figsize=(14, 30))
g = sns.clustermap(
    heatmap_data,
    cmap='RdBu_r', center=0, vmin=-2, vmax=2,
    dendrogram_ratio=(0.2, 0.05),
    row_linkage=row_linkage,
    col_linkage=col_linkage,
    figsize=(14, 30),
    cbar_kws={'label': 'log2FoldChange'}
)
g.savefig(f'{plots_path}/hormoneSignalling/hormone_signalling_joint_ar_targets_deg_l4_filtered_heatmap.pdf', dpi=300)

In [ ]:
# Pairwise correlation of cell types based on heatmap_data
corr = heatmap_data.corr(method='pearson')

# Clustered correlation clustermap
g = sns.clustermap(
    corr,
    cmap='vlag',
    center=0,
    linewidths=0.5,
    figsize=(12, 12),
    cbar_kws={'label': 'Pearson r'}
)
g.fig.suptitle('Pairwise correlation between cell types', y=1.02)
g.savefig(f'{plots_path}/hormoneSignalling/celltype_pairwise_correlation_clustermap.pdf', dpi=300, bbox_inches='tight')
plt.close(g.fig)

# # Plain heatmap for quick inspection
# plt.figure(figsize=(12, 10))
# sns.heatmap(corr, cmap='vlag', center=0, square=True, cbar_kws={'label': 'Pearson r'})
# plt.title('Pairwise correlation between cell types')
# plt.tight_layout()
# plt.savefig(f'{plots_path}/hormoneSignalling/celltype_pairwise_correlation_heatmap.pdf', dpi=300)
# plt.show()

In [ ]:
from sklearn.decomposition import PCA

# PCA on cell types (rows = cell types, columns = genes)
X = heatmap_data.T.fillna(0)

pca = PCA(n_components=2)
pcs = pca.fit_transform(X)

pca_df = pd.DataFrame(
    pcs,
    index=X.index,
    columns=["PC1", "PC2"]
)

plt.figure(figsize=(10, 8))
plt.scatter(pca_df["PC1"], pca_df["PC2"], s=60)

for name, row in pca_df.iterrows():
    plt.text(row["PC1"], row["PC2"], name, fontsize=9, ha="left", va="bottom")

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of cell types based on heatmap data")
plt.axhline(0, color="grey", lw=0.5)
plt.axvline(0, color="grey", lw=0.5)
plt.tight_layout()
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

W = rbf_kernel(heatmap_data.values, gamma=20)
D = np.diag(np.sum(W, axis=1))

L = D - W

eigenvalues, eigenvectors = np.linalg.eigh(L)

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(np.arange(len(eigenvalues)), eigenvalues, s=8)
plt.xlabel("Index")
plt.ylabel("Eigenvalue")
plt.title("Eigenvalues from Laplacian matrix")
plt.grid(True, alpha=0.3)
plt.tight_layout()
#plt.savefig(f'{plots_path}/hormoneSignalling/laplacian_eigenvalues.pdf', dpi=300)
plt.show()

In [ ]:
from sklearn.cluster import SpectralCoclustering
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use the existing heatmap_data (genes x cell_types)
data = heatmap_data.values
n_rows, n_cols = data.shape

# Global SSE for baseline
global_mean = data.mean()
global_sse = ((data - global_mean) ** 2).sum()

# Search range for n_clusters
max_k = min(100, n_cols - 1, n_rows - 1)
ks = list(range(5, max_k + 1, 5))
seeds = [0, 1, 2, 3]  # average across seeds for stability

scores = []
for k in ks:
    explained_vals = []
    for seed in seeds:
        model = SpectralCoclustering(n_clusters=k, random_state=seed)
        model.fit(data)
        rlab = model.row_labels_
        clab = model.column_labels_
        sse_inside = 0.0
        for ri in range(k):
            rows_mask = (rlab == ri)
            if not rows_mask.any():
                continue
            for cj in range(k):
                cols_mask = (clab == cj)
                if not cols_mask.any():
                    continue
                block = data[np.ix_(rows_mask, cols_mask)]
                mu = block.mean()
                sse_inside += ((block - mu) ** 2).sum()
        explained = 1 - sse_inside / global_sse
        explained_vals.append(explained)
    scores.append(np.mean(explained_vals))

scores_df = pd.DataFrame({"n_clusters": ks, "explained_fraction": scores})
print(scores_df)

# Choose best k (max explained fraction)
best_k = int(scores_df.loc[scores_df["explained_fraction"].idxmax(), "n_clusters"])
print(f"Selected n_clusters = {best_k}")

# Fit final model and visualize rearranged matrix
model = SpectralCoclustering(n_clusters=best_k, random_state=0)
model.fit(data)
fit_data = data[np.argsort(model.row_labels_)][:, np.argsort(model.column_labels_)]

plt.figure(figsize=(10, 10))
plt.matshow(fit_data, cmap="RdBu_r", fignum=1)
plt.title(f"Spectral Coclustering (n_clusters={best_k}) - rearranged")
plt.colorbar(label="log2FoldChange")
plt.tight_layout()
plt.savefig(f"{plots_path}/hormoneSignalling/spectral_coclustering_n{best_k}_rearranged.pdf", dpi=300)
plt.show()

# Plot explained_fraction vs n_clusters
plt.figure(figsize=(6, 4))
plt.plot(scores_df["n_clusters"], scores_df["explained_fraction"], marker="o")
plt.xlabel("n_clusters")
plt.ylabel("Explained fraction (1 - SSE_inside / global_SSE)")
plt.title("Model fit vs number of biclusters")
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{plots_path}/hormoneSignalling/spectral_coclustering_explained_vs_k.pdf", dpi=300)
plt.show()

In [ ]:
from sklearn.cluster import SpectralCoclustering

# Spectral coclustering on genes x cell_types (data)
n_coclusters = 35
cocluster_model = SpectralCoclustering(n_clusters=n_coclusters, random_state=0)
cocluster_model.fit(heatmap_data)

# Get row and column labels
row_labels = cocluster_model.row_labels_
col_labels = cocluster_model.column_labels_

# Rearrange matrix by cluster labels
fit_data_coclustered = data[np.argsort(row_labels)][:, np.argsort(col_labels)]

# Visualize rearranged coclustered matrix
plt.figure(figsize=(10, 12))
plt.imshow(fit_data_coclustered, cmap='RdBu_r', aspect='auto')
plt.title(f'Spectral Coclustering (n_clusters={n_coclusters}) - Rearranged')
plt.colorbar(label='log2FoldChange')
plt.xlabel('Cell Types')
plt.ylabel('Genes')
plt.tight_layout()
plt.savefig(f'{plots_path}/hormoneSignalling/spectral_coclustering_8_rearranged.pdf', dpi=300)
plt.show()

print(f"Gene clusters: {len(set(row_labels))}")
print(f"Cell type clusters: {len(set(col_labels))}")


In [ ]:
# PCA of cell types colored by coclustering cluster label
cell_clusters = pd.Series(cocluster_model.column_labels_, index=heatmap_data.columns, name="cluster")

pca_plot = pca_df.join(cell_clusters)

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=pca_plot,
    x="PC1",
    y="PC2",
    hue="cluster",
    palette="tab20",
    s=70,
)

for name, row in pca_plot.iterrows():
    plt.text(row["PC1"], row["PC2"], name, fontsize=8, ha="left", va="bottom")

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of cell types colored by cluster label")
plt.axhline(0, color="grey", lw=0.5)
plt.axvline(0, color="grey", lw=0.5)
plt.legend(title="cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
cluster_labels

In [ ]:
cocluster_model.get_indices(0)

In [ ]:
clusters = {}
for c in range(0,35):
    clusters[c] = ([heatmap_data.index[i] for i in cocluster_model.get_indices(c)[0]],
                    [heatmap_data.columns[i] for i in cocluster_model.get_indices(c)[1]])

In [ ]:
import pprint

pprint.pprint(clusters, compact=True, width = 200)

In [ ]:
gene2module = pd.concat([pd.DataFrame(data={'module': i, 'gene_name': m[0]}) for i, m in clusters.items()], ignore_index=True)
gene2module

In [ ]:
deg_modules = gene2module.merge(degs_ar, on='gene_name', how='left')
deg_modules = deg_modules.groupby(['cell_type','module'], observed=True).agg(
    n_genes = ('gene_name', 'count'),
    n_deg = ('padj', lambda x: (x < 0.05).sum()),
    mean_log2FC = ('log2FoldChange', 'mean')
).reset_index()
deg_modules.sort_values('n_deg', ascending=False)

In [ ]:
# Create a pivot table for the heatmap
heatmap_pivot = deg_modules.pivot_table(
    index='module',
    columns='cell_type',
    values='mean_log2FC',
    fill_value=0
)

# Create clustered heatmap
plt.figure(figsize=(10,8))
g = sns.clustermap(
    heatmap_pivot,
    cmap='RdBu_r',
    center=0,
    vmin=-2,
    vmax=2,
    linewidths=0.5,
    cbar_kws={'label': 'Mean log2FoldChange'},
    figsize=(10,8)
)
g.ax_heatmap.tick_params(axis='y', labelrotation=0)
g.fig.suptitle('Module-CellType heatmap colored by mean log2FoldChange', y=0.98)
g.savefig(f'{plots_path}/hormoneSignalling/deg_modules_celltype_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()

https://claude.ai/chat/7d4c20d1-170c-4f7c-aa82-f0ff99fea35e